In [1]:
import pandas as pd

df = pd.read_csv("../datasets/raw/esol.csv")

print(df.shape)
df.head()

(1128, 10)


,Compound ID,ESOL predicted log solubility in mols per litre,Minimum Degree,Molecular Weight,Number of H-Bond Donors,Number of Rings,Number of Rotatable Bonds,Polar Surface Area,measured log solubility in mols per litre,smiles
0,Amigdalin,-0.974,1,457.432,7,3,7,202.32,-0.77,OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)...
1,Fenfuram,-2.885,1,201.225,1,2,2,42.24,-3.30,Cc1occc1C(=O)Nc2ccccc2
2,citral,-2.579,1,152.237,0,0,4,17.07,-2.06,CC(C)=CCCC(C)=CC(=O)
3,Picene,-6.618,2,278.354,0,5,0,0.00,-7.87,c1ccc2c(c1)ccc3c2ccc4c5ccccc5ccc43
4,Thiophene,-2.232,2,84.143,0,1,0,0.00,-1.33,c1ccsc1


In [2]:
print(df.columns.tolist())

['Compound ID', 'ESOL predicted log solubility in mols per litre', 'Minimum Degree', 'Molecular Weight', 'Number of H-Bond Donors', 'Number of Rings', 'Number of Rotatable Bonds', 'Polar Surface Area', 'measured log solubility in mols per litre', 'smiles']


In [3]:
from rdkit import Chem
from rdkit.Chem import Descriptors

def extract_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    return {
        "MolWt": Descriptors.MolWt(mol),
        "MolLogP": Descriptors.MolLogP(mol),
        "TPSA": Descriptors.TPSA(mol),
        "NumHDonors": Descriptors.NumHDonors(mol),
        "NumHAcceptors": Descriptors.NumHAcceptors(mol),
        "NumRotatableBonds": Descriptors.NumRotatableBonds(mol),
        "RingCount": Descriptors.RingCount(mol)
    }

In [4]:
descriptor_data = []

for smiles in df["smiles"]:
    descriptor_data.append(extract_descriptors(smiles))

descriptor_df = pd.DataFrame(descriptor_data)

descriptor_df.head()

,MolWt,MolLogP,TPSA,NumHDonors,NumHAcceptors,NumRotatableBonds,RingCount
0,457.432,-3.10802,202.32,7,12,7,3
1,201.225,2.84032,42.24,1,2,2,2
2,152.237,2.87800,17.07,0,1,4,0
3,278.354,6.29940,0.00,0,0,0,5
4,84.143,1.74810,0.00,0,1,0,1


In [5]:
target = df["measured log solubility in mols per litre"]

data = descriptor_df.copy()
data["Solubility"] = target

data.head()

,MolWt,MolLogP,TPSA,NumHDonors,NumHAcceptors,NumRotatableBonds,RingCount,Solubility
0,457.432,-3.10802,202.32,7,12,7,3,-0.77
1,201.225,2.84032,42.24,1,2,2,2,-3.30
2,152.237,2.87800,17.07,0,1,4,0,-2.06
3,278.354,6.29940,0.00,0,0,0,5,-7.87
4,84.143,1.74810,0.00,0,1,0,1,-1.33


In [6]:
print(data.isnull().sum())

MolWt                0
MolLogP              0
TPSA                 0
NumHDonors           0
NumHAcceptors        0
NumRotatableBonds    0
RingCount            0
Solubility           0
dtype: int64


In [7]:
X = data.drop("Solubility", axis=1)
y = data["Solubility"]

print(X.shape)
print(y.shape)

(1128, 7)
(1128,)


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(902, 7)
(226, 7)


In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.shape)

(902, 7)


In [10]:
print(X_train_scaled.shape)
print(X_test_scaled.shape)

(902, 7)
(226, 7)


In [11]:
from sklearn.decomposition import PCA

pca = PCA(n_components=4)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(X_train_pca.shape)
print(X_test_pca.shape)

(902, 4)
(226, 4)


In [12]:
print(
    "Explained Variance:",
    pca.explained_variance_ratio_.sum()
)

Explained Variance: 0.9566014787672797


In [ ]:
import pennylane as qml
import torch
import torch.nn as nn

In [2]:
n_qubits = 4

dev = qml.device(
    "default.qubit",
    wires=n_qubits
)

In [3]:
@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):

    qml.AngleEmbedding(
        inputs,
        wires=range(n_qubits)
    )

    qml.StronglyEntanglingLayers(
        weights,
        wires=range(n_qubits)
    )

    return [
        qml.expval(qml.PauliZ(i))
        for i in range(n_qubits)
    ]

In [4]:
weight_shapes = {
    "weights": (3, n_qubits, 3)
}

In [5]:
qml_layer = qml.qnn.TorchLayer(
    quantum_circuit,
    weight_shapes
)